<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<font size="5" color='grey'> <b>
SQL RAG mit Chat-Historie
</b></font> </br>


---

In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY', 'HF_TOKEN'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()
# LangSmith Tracing
run_cfg = {
    "run_name": "M09_SQL_RAG",
    "tags": ["m09", "sql_rag"],
    "metadata": {"notebook": "M09", "version": "1.0"}
}


✓ OPENAI_API_KEY erfolgreich gesetzt
✓ HF_TOKEN erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.0.2
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 34.125.112.168
Hostname: 168.112.125.34.bc.googleusercontent.com
Stadt: Las Vegas
Region: Nevada
Land: US
Koordinaten: 36.1750,-115.1372
Provider: AS396982 Google LLC
Postleitzahl: 89111
Zeitzone: America/Los_Angeles


In [2]:
#@title 📂 Datenbank { display-mode: "form" }
from genai_lib.utilities import copy_from_github

# Northwind-Datenbank herunterladen
copy_from_github(
    source="ralf-42/GenAI/02_daten/05_sonstiges",
    target=".",
    mask="northwind.db",
)

  Kopiere: 02_daten/05_sonstiges/northwind.db → northwind.db

Ergebnis: 1 Datei(en) kopiert → .


['northwind.db']

# 1 | Einführung in SQL RAG
---

SQL RAG verbindet Large Language Models mit Datenbankabfragen: Natürlichsprachliche Anfragen werden in SQL übersetzt, ausgeführt und die Ergebnisse in verständliche Antworten zurückübersetzt. Das überbrückt die Lücke zwischen menschlicher Sprache und strukturierten Daten — Datenbankschema-Analyse, SQL-Generierung und Ergebnisinterpretation laufen als integrierte Pipeline.

> [!WARNING] Einschränkung:<br>
> SQL RAG funktioniert gut für klar strukturierte Schemas. Mehrdeutige Spaltennamen, fehlende Primärschlüssel-Dokumentation oder sehr komplexe Joins führen dagegen häufig zu fehlerhaften SQL-Abfragen.

<p><font color='black' size="5">
Warum SQL für RAG?
</font></p>

SQL löst vier typische Herausforderungen von RAG-Systemen: **Komplexer Datenabruf** — SQL-Abfragen filtern gezielt aus strukturierten Datensätzen, die unstrukturierte Retrieval-Mechanismen nicht sauber trennen können. **Datenqualität** — Filtern und Ranking nach Zeitstempel, Kategorie oder Relevanzwert lassen sich direkt in der Query kodieren. **Skalierbarkeit** — relationale Datenbanken sind für große, strukturierte Datensätze optimiert; der Retrieval-Mechanismus skaliert ohne Neuindexierung. **Echtzeit** — Query-Caching und Indizierung reduzieren Latenz auf ein Niveau, das für interaktive Anwendungen geeignet ist.

Für unstrukturierte Quellen wie Textdokumente oder Wissensgraphen ist Embedding-basiertes RAG in der Regel besser geeignet.


# 2 | Vergleich SQL RAG vs RAG
---

Während sowohl SQL RAG als auch RAG (Retrieval-Augmented Generation) die Fähigkeiten von LLMs erweitern, gibt es wichtige Unterschiede:



| Merkmal         | SQL RAG      | Retrieval-Augmented Generation (RAG)    |
| --------------- | ------------------------------------ | --------------------------------------- |
| Datenquelle     | Strukturierte Datenbanken            | Textdokumente, Wissensbasen             |
| Abfragemethode  | SQL-Generierung                      | Semantische Suche, Embedding-Vergleiche |
| Datenstruktur   | Schema-basiert, tabellarisch         | Unstrukturiert oder semi-strukturiert   |
| Genauigkeit     | Präzise durch Datenbankintegrität    | Abhängig von der Retrieval-Qualität     |
| Anwendungsfälle | Geschäftsanalysen, Berichterstellung | Dokumentensuche, Wissensbasis-Anfragen  |
| Aktualisierung  | In Echtzeit durch aktuelle DB-Daten  | Erfordert Neuindexierung bei Änderungen |



SQL RAG eignet sich besonders für Szenarien, in denen präzise, aktuelle Daten benötigt werden, während RAG Stärken bei der Verarbeitung großer Textmengen hat.



# 3 | Integration LLM und DB
---



Die Integration von LLMs mit Datenbanken erfolgt über mehrere Komponenten:

1. **Schema-Analyse**: Das LLM muss das Datenbankschema verstehen (Tabellen, Spalten, Beziehungen)
2. **Anfrage-Übersetzung**: Umwandlung der natürlichsprachlichen Anfrage in SQL
3. **Abfrage-Ausführung**: Verbindung zur Datenbank und Ausführung der generierten SQL-Abfrage
4. **Ergebnis-Interpretation**: Analyse und Interpretation der Abfrageergebnisse

<img src="https://raw.githubusercontent.com/ralf-42/GenAI/main/07_image/sql_rag_process.png" width="750" alt="Avatar">


In [3]:
# ── Stdlib ────────────────────────────────────────────────────────────────────
import traceback

# ── Third-party ───────────────────────────────────────────────────────────────
import gradio as gr

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# ── Pydantic ──────────────────────────────────────────────────────────────────
from pydantic import BaseModel, Field

# ── Modell-Konfiguration ──────────────────────────────────────────────────────
from genai_lib.model_config import BASELINE

# 1. Datenbankverbindung herstellen
db = SQLDatabase.from_uri("sqlite:///northwind.db")

# 2. LLM initialisieren (Kurznotation: "provider:model")
llm = init_chat_model(BASELINE)

# 3. Schema laden und cachen
db_schema = db.get_table_info()

In [4]:
# 3. Gecachtes Schema anzeigen (kein erneuter DB-Aufruf nötig)
print(db_schema)


CREATE TABLE "Categories" (
	"CategoryID" INTEGER, 
	"CategoryName" TEXT, 
	"Description" TEXT, 
	"Picture" BLOB, 
	PRIMARY KEY ("CategoryID")
)

/*
3 rows from Categories table:
CategoryID	CategoryName	Description	Picture
1	Beverages	Soft drinks, coffees, teas, beers, and ales	b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x02\x00\x00d\x00d\x00\x00\xff\xec\x00\x11Ducky\x00\x01\x00\x0
2	Condiments	Sweet and savory sauces, relishes, spreads, and seasonings	b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x02\x00\x00d\x00d\x00\x00\xff\xec\x00\x11Ducky\x00\x01\x00\x0
3	Confections	Desserts, candies, and sweet breads	b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x02\x00\x00d\x00d\x00\x00\xff\xec\x00\x11Ducky\x00\x01\x00\x0
*/


CREATE TABLE "CustomerCustomerDemo" (
	"CustomerID" TEXT NOT NULL, 
	"CustomerTypeID" TEXT NOT NULL, 
	PRIMARY KEY ("CustomerID", "CustomerTypeID"), 
	FOREIGN KEY("CustomerTypeID") REFERENCES "CustomerDemographics" ("CustomerTypeID"), 
	FOREIGN KEY("CustomerID") REFERENCES "Customers" ("Cu

In [5]:
# 4. Natürlichsprachliche Anfrage
user_query = "Wie viele Mitarbeiter haben wir?"

Die Herausforderung liegt in der korrekten Interpretation des Schemas und der präzisen Übersetzung der Anfragen.



# 4 | SQL-Generierung mit LLMs
---



Die SQL-Generierung ist ein kritischer Bestandteil von SQL RAG und erfolgt in mehreren Schritten:

1. **Prompt-Engineering**: Entwicklung spezifischer Prompts, die das Datenbankschema und die Anforderungen enthalten
2. **Query-Planung**: Analyse der Anfrage, um die benötigten Tabellen und Joins zu identifizieren
3. **SQL-Syntax-Generierung**: Erzeugung syntaktisch korrekter SQL-Abfragen
4. **Validierung**: Überprüfung der generierten Abfrage vor der Ausführung

In [6]:
# Pydantic-Model für strukturierte SQL-Ausgabe
class SQLQuery(BaseModel):
    """Strukturierte SQL-Abfrage."""
    sql: str = Field(description="Reine SQL-Abfrage (nur SELECT, ohne Markdown/Formatierung)")

In [7]:
# SQL-Prompt als Markdown-Template laden (05_prompt/)
sql_prompt = load_prompt(
    "https://github.com/ralf-42/GenAI/blob/main/05_prompt/m09_sql_prompt.md"
)

# with_structured_output
sql_llm = llm.with_structured_output(SQLQuery)

In [8]:
# Chain
sql_generator = (
    RunnablePassthrough.assign(schema=lambda _: db_schema)
    | sql_prompt
    | sql_llm
)

In [9]:
# Verwendung - Ergebnis ist ein Pydantic-Objekt
result = sql_generator.invoke({"query": user_query}, config=run_cfg)
sql_query = result.sql
print(f"Generierte SQL: {sql_query}")
print(f"Typ: {type(result)}")  # SQLQuery

Generierte SQL: SELECT COUNT(*) AS MitarbeiterAnzahl
FROM Employees;
Typ: <class '__main__.SQLQuery'>


# 5 | Hands-On: SQL RAG `northwind.db`
---


LangChain bietet leistungsstarke Tools für die Implementierung von SQL RAG-Lösungen:

1. **SQLDatabase**: Verbindung zur Datenbank mit `db.run()` Methode
2. **ChatPromptTemplate**: Strukturierte Prompts mit System/Human Messages  
3. **LCEL**: LangChain Expression Language für Chains
4. **SQL Agent** (optional): Intelligente Agents für komplexe Queries


<p><font color='black' size="5">
Erläuterung des SQL RAG-Beispiels
</font></p>

**Was passiert hier?**

Das Beispiel baut eine SQL RAG-Anwendung mit sieben Bausteinen auf: Northwind-Datenbank über SQLite, LLM-Anbindung via `init_chat_model()`, typsichere SQL-Generierung mit `with_structured_output(SQLQuery)`, einmalig gecachtes DB-Schema, SQL-Validierung direkt in `execute_query()`, vorberechnete Chains (`sql_generator`, `analysis_chain`) und ein Gradio-Chatinterface mit Histoire.

Der Ablauf bei einer Nutzeranfrage: `sql_generator` erzeugt ein `SQLQuery`-Objekt → `execute_query()` validiert und führt aus → `analysis_chain` interpretiert die Ergebnisse → Antwort wird ausgegeben.

<p><font color='black' size="5">
💬 Chat-Historie
</font></p>

Das System unterstützt **kontextbewusste Konversationen**: Gradio speichert die letzten Konversationen automatisch, die letzten drei Frage-Antwort-Paare werden als `{history_text}` in den Prompt eingebettet. Das LLM nutzt diesen Kontext für Folgefragen — etwa "Zeige mehr Details zu diesen Produkten" ohne Wiederholung der vorherigen Ergebnisse.

**Beispiel-Konversation:**
```
👤 User: "Welche Produkte sind nicht auf Lager?"
🤖 Bot: [Zeigt 3 Produkte: Chai, Chang, Gorgonzola]

👤 User: "Zeige mir mehr Details zu diesen Produkten"
🤖 Bot: [Generiert SQL mit WHERE-Klausel für die 3 Produkte]

👤 User: "Welche Lieferanten haben diese Produkte geliefert?"
🤖 Bot: [JOIN mit Suppliers-Tabelle basierend auf Kontext]
```

Token-Optimierung: Nur die letzten 3 Einträge werden eingebettet, ältere werden verworfen.

[DatenbankSchema](https://github.com/jpwhite3/northwind-SQLite3/blob/main/docs/Northwind_ERD.png)


[Quelle](https://github.com/jpwhite3/northwind-SQLite3)

In [10]:
#@markdown   <p><font size="4" color='green'> SQL-RAG-Prozess</font> </br></p>

# Parametrisierte Funktion
def show_architecture(highlight=None):
    """Zeigt die SQL-RAG Architektur mit optionaler Hervorhebung.

    Args:
        highlight: None, "sql", "output" oder "ui"
    """
    styles = {
        "sql": "\n    style SQL fill:#4a90d9,stroke:#2d5a87,color:#fff",
        "output": "\n    style Output fill:#4a90d9,stroke:#2d5a87,color:#fff",
        "ui": "\n    style Gradio fill:#4a90d9,stroke:#2d5a87,color:#fff\n    style RESULT fill:#4a90d9,stroke:#2d5a87,color:#fff",
    }

    diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    USER["User"] --> Gradio["Gradio: chatbot_response()"]

    subgraph SQL["<b>SQL-Abfrage</b>"]
        direction TB
        schema_cache["db_schema (cached)"]
        sql_gen["sql_generator (structured output)"]
        validate["validate_sql()"]
        exec["execute_query()"]
        schema_cache --> sql_gen --> validate --> exec
    end

    subgraph Output["<b>Erstellung Output </b>"]
        direction TB
        analyze["analysis_chain"]
    end

    Gradio --> schema_cache
    Gradio --> LLM["LLM"]
    exec --> analyze
    analyze --> RESULT["Gradio: Antwort"]

    DB[(northwind.db)] -.-> |"Schema (einmalig)"| schema_cache
    exec -.-> |"SQL"| DB
    DB -.-> |"Ergebnis"| exec

    LLM -.-> sql_gen
    LLM -.-> analyze
""" + styles.get(highlight, "")

    mermaid(diagram, width=1100, height=400)

# Gesamtübersicht
show_architecture()

<p><font color='black' size="5">
Programm
</font></p>

In [11]:
# Relativer Pfad
DB_PATH = "northwind.db"
DB_URI = f"sqlite:///{DB_PATH}"

In [12]:
# LLM
llm = init_chat_model(BASELINE)

# SQL-Datenbank initialisieren
db = SQLDatabase.from_uri(DB_URI)

# Schema einmalig laden und cachen
db_schema = db.get_table_info()
print(f"Schema geladen: {len(db_schema)} Zeichen")

Schema geladen: 8910 Zeichen


<p><font color='black' size="5">
SQL & Analyse Prompts
</font></p>

Die Prompts sind als **Markdown-Templates** in `05_prompt/` ausgelagert und werden via `load_prompt()` von GitHub geladen:

- `m09_sql_rag_generator.md` — SQL-Generierung mit Historie-Kontext
- `m09_sql_rag_analysis.md` — Business-Analyse der Ergebnisse

In [13]:
# Prompt als Markdown-Template ausgelagert (05_prompt/)
sql_prompt = load_prompt(
    "https://raw.githubusercontent.com/ralf-42/GenAI/main/05_prompt/m09_sql_rag_generator.md"
)
print(f"SQL-Prompt geladen: {len(sql_prompt.messages)} Messages")

SQL-Prompt geladen: 2 Messages


In [14]:
# Prompt als Markdown-Template ausgelagert (05_prompt/)
analysis_prompt = load_prompt(
    "https://raw.githubusercontent.com/ralf-42/GenAI/main/05_prompt/m09_sql_rag_analysis.md"
)
print(f"Analyse-Prompt geladen: {len(analysis_prompt.messages)} Messages")

Analyse-Prompt geladen: 2 Messages


<p><font color='black' size="5">
SQL-Abfrage
</font></p>

In [15]:
#@markdown   <p><font size="4" color='green'> SQL-RAG-Prozess</font> </br></p>

show_architecture(highlight="sql")

In [16]:
# with_structured_output: LLM gibt direkt ein SQLQuery-Objekt zurück
sql_llm = llm.with_structured_output(SQLQuery)

In [17]:
# Schema aus Cache + structured output
sql_generator = (
    RunnablePassthrough.assign(schema=lambda _: db_schema)
    | sql_prompt
    | sql_llm
)

# Analysis-Chain definieren
analysis_chain = analysis_prompt | llm | StrOutputParser()

In [18]:
# SQL-Validierung direkt in execute_query integriert
def validate_sql(sql_query: str) -> tuple[bool, str]:
    """Validiert eine SQL-Abfrage auf Sicherheit."""
    if not sql_query.strip().upper().startswith("SELECT"):
        return False, "Nur SELECT-Anweisungen sind erlaubt."

    dangerous_commands = ["DROP", "DELETE", "TRUNCATE", "UPDATE", "INSERT", "ALTER"]
    for command in dangerous_commands:
        if f" {command} " in sql_query.upper():
            return False, f"Unerlaubter SQL-Befehl: {command}"

    return True, "OK"


def execute_query(sql_query: str) -> str:
    """Validiert und führt eine SQL-Abfrage aus."""
    # Sicherheitsprüfung vor Ausführung
    is_valid, message = validate_sql(sql_query)
    if not is_valid:
        return f"⚠️ Sicherheitsfehler: {message}"

    try:
        result = db.run(sql_query)
        if not result or result.strip() == "":
            return "Keine Ergebnisse gefunden."
        return result
    except Exception as e:
        return f"Fehler bei der Ausführung: {str(e)}"

<p><font color='black' size="5">
Erstellung Output
</font></p>

In [19]:
#@markdown   <p><font size="4" color='green'> SQL-RAG-Prozess</font> </br></p>

show_architecture(highlight="output")

In [20]:
# Historie-Formatierung als eigene Funktion
def format_history(history) -> str:
    """Formatiert Gradio-Historie für LLM-Kontext (letzte 3 Einträge)."""
    if not history:
        return ""

    parts = []
    for i, (user_msg, bot_msg) in enumerate(history[-3:], 1):
        parts.append(f"[Vorherige Frage {i}]: {user_msg}")
        if bot_msg and "### Analyse" in bot_msg:
            analysis_part = bot_msg.split("### Analyse")[-1].strip()
            parts.append(f"[Vorherige Antwort {i}]: {analysis_part[:300]}")
    return "\n".join(parts)


# analysis_chain
def analyze_results(query, sql_query, results, history_text=""):
    """Analysiert die Ergebnisse und gibt eine natürlichsprachliche Antwort zurück."""
    return analysis_chain.invoke({
        "query": query,
        "sql_query": sql_query,
        "results": results,
        "history_text": history_text
    }, config=run_cfg)

<p><font color='black' size="5">
UI Gradio In/Out
</font></p>

In [21]:
#@markdown   <p><font size="4" color='green'> SQL-RAG-Prozess</font> </br></p>

show_architecture(highlight="ui")

In [22]:
def chatbot_response(message, history):
    """Verarbeitet Benutzeranfragen mit optimierter SQL-RAG Pipeline."""
    try:
        # Historie formatieren (ausgelagerte Funktion)
        history_text = format_history(history)

        # SQL generieren (structured output - kein Regex nötig!)
        result = sql_generator.invoke({
            "query": message,
            "history_text": history_text
        }, config=run_cfg)
        sql_query = result.sql  # Pydantic-Objekt → sauberer String

        # Debug-Ausgabe
        print(f"Generierte SQL: {sql_query}")

        # Abfrage Datenbank (mit integrierter Validierung)
        results = execute_query(sql_query)

        # Analysiere die Ergebnisse (vordefinierte Chain)
        analysis = analyze_results(message, sql_query, results, history_text)

        # Antwort formatieren
        return f"""### Deine Anfrage
{message}

### SQL-Abfrage
```sql
{sql_query}
```

### Ergebnisse
{results}

### Analyse
{analysis}"""

    except Exception as e:
        error_details = traceback.format_exc()
        print(f"Fehler Details:\n{error_details}")
        return f"Ein Fehler ist aufgetreten: {str(e)}\n\nDetails siehe Console-Output."

In [23]:
# Beispielfragen für Gradio-Interface definieren
example_questions = [
    "Welche Produkte sind aktuell nicht mehr auf Lager? Nenne die Top 3.",
    "Welche Bestellung von welchem Kunden hatte den höchsten Gesamtwert? Nenne die Top 3.",
    "Aus welchen Ländern stammen die meisten Kunden? Nenne die Top 3.",
    "Sind alle Artikel der Bestellung der Rattlesnake Canyon Grocery vom 1998-05-06 in ausreichender Anzahl auf Lager?",
]

In [24]:
# Gradio Interface erstellen
demo = gr.ChatInterface(
    fn=chatbot_response,
    title="📚 SQL RAG",
    description="""**Features:**
- 💬 **Chat-Historie**: Stelle Folge-Fragen basierend auf vorherigen Antworten
- 🤖 **Structured Output**: SQL-Generierung über `with_structured_output()` (kein Regex)
- 🔒 **Integrierte Validierung**: SQL-Sicherheitsprüfung vor jeder Ausführung
- ⚡ **Schema-Caching**: DB-Schema einmalig geladen (Performance)
- 📊 **Intelligente Analyse**: Automatische Interpretation der Ergebnisse

""",
    examples=example_questions,
)

<p><font color='black' size="5">
Starten der App
</font></p>

**Beispiel-Fragen:**



+ Gib die Artikelliste für die Bestellung 11031 mit Einzelpreis und Gesamtpreis aus, wobei sich der Gesamtpreis aus der Anzahl und dem Einzelpreis ergibt.
+ Welcher Mitarbeiter ist für die Bestellung mit der Nummer 10266 zuständig?
+ Über welche Versandfirma wurde die Bestellung 10266 ausgeliefert?
+ Sind alle Artikel der Bestellung der Rattlesnake Canyon Grocery vom 1998-05-06 in ausreichender Anzahl auf Lager?
+ Welche Kunden haben schon Artikel der Firma 'Escargots Nouveaux' gekauft?




In [25]:
# App starten
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7d9a997608f6501a22.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



# 6 | Validierung und Sicherheit
---



Die Sicherheit ist bei der Arbeit mit datenbankgesteuerten Anwendungen von entscheidender Bedeutung. SQL RAG-Implementierungen müssen folgende Sicherheitsaspekte berücksichtigen:

1. **SQL-Injection-Prävention**:
    
    - Validierung und Bereinigung generierter SQL-Abfragen
    - Verwendung von parametrisierten Abfragen
    - Beschränkung der SQL-Befehle (z.B. nur SELECT-Anweisungen zulassen)
2. **Zugriffskontrolle**:
    
    - Verwendung von Datenbanknutzern mit eingeschränkten Rechten
    - Zugriffsbeschränkungen auf bestimmte Tabellen oder Ansichten
    - Implementierung von Row-Level-Security
3. **Datenvalidierung**:
    
    - Überprüfung der generierten SQL-Abfragen auf verdächtige Muster
    - Begrenzung der Abfragekomplexität und -länge
    - Timeouts für lang laufende Abfragen


In [26]:
# Test mit verschiedenen Abfragen (nutzt validate_sql() aus §5)
print("SELECT-Test:", validate_sql("SELECT * FROM Products"))
print("DROP-Test:  ", validate_sql("DROP TABLE Products"))
print("DELETE-Test:", validate_sql("DELETE FROM Products WHERE id=1"))

SELECT-Test: (True, 'OK')
DROP-Test:   (False, 'Nur SELECT-Anweisungen sind erlaubt.')
DELETE-Test: (False, 'Nur SELECT-Anweisungen sind erlaubt.')


Eine gründliche Validierung vor der Ausführung ist entscheidend für die Sicherheit der Anwendung.


# 7 | Praktische Anwendungsfälle
---

SQL RAG eignet sich für Szenarien, in denen präzise, aktuelle Daten aus strukturierten Quellen benötigt werden: Business-Intelligence-Dashboards mit natürlichsprachlichen Abfragen, Datenanalyse für Nicht-Techniker ohne SQL-Kenntnisse, automatisierte Berichterstellung und Kundenservice-Anwendungen mit schnellem Datenbankzugriff.

Der größte Mehrwert liegt in der **Demokratisierung des Datenzugriffs**: Fachabteilungen können direkt mit Unternehmensdaten arbeiten, ohne SQL zu beherrschen. Die Grenze liegt bei unstrukturierten Quellen — dort ist Embedding-basiertes RAG die bessere Wahl.

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> Generative KI darf und soll im Kurs auch als Lernunterstützung genutzt werden — z. B. Gemini in Google Colab, um Fehlermeldungen zu verstehen, Teilschritte zu klären oder Code-Varianten zu prüfen.
> Der Schwerpunkt des Kurses bleibt: GenAI-Apps selbst verstehen, aufbauen und gezielt weiterentwickeln.

**Grundlagen**

Beantworte drei einfache SELECT-Fragen (z.B. "Welche Kunden gibt es?") über den SQL-Agenten in natürlicher Sprache.

**✅ Erledigt wenn:** Alle drei SQL-Abfragen liefern Ergebnisse, die manuell gegen die Datenbank geprüft und korrekt befunden wurden.

In [27]:
# Grundlagen: 3 einfache SELECT-Fragen über die SQL-RAG-Pipeline
# Startpunkt: sql_generator + execute_query aus Kapitel 4/5

fragen = [
    "Welche Kunden gibt es in Deutschland?",
    "Welche Produkte kosten mehr als 30 Euro?",
    "Wie viele Bestellungen wurden insgesamt aufgegeben?",
]

for frage in fragen:
    result = sql_generator.invoke({"query": frage, "history_text": ""}, config=run_cfg)
    print(f"Frage: {frage}")
    print(f"SQL:   {result.sql}")
    print(f"Ergebnis: {execute_query(result.sql)}")
    print('---')


Frage: Welche Kunden gibt es in Deutschland?
SQL:   SELECT c.CustomerID,
       c.CompanyName,
       c.ContactName,
       c.City,
       c.Region,
       c.PostalCode,
       c.Country,
       c.Phone
FROM Customers c
WHERE c.Country = 'Germany'
ORDER BY c.CompanyName
LIMIT 10;
Ergebnis: [('ALFKI', 'Alfreds Futterkiste', 'Maria Anders', 'Berlin', 'Western Europe', '12209', 'Germany', '030-0074321'), ('BLAUS', 'Blauer See Delikatessen', 'Hanna Moos', 'Mannheim', 'Western Europe', '68306', 'Germany', '0621-08460'), ('WANDK', 'Die Wandernde Kuh', 'Rita Müller', 'Stuttgart', 'Western Europe', '70563', 'Germany', '0711-020361'), ('DRACD', 'Drachenblut Delikatessen', 'Sven Ottlieb', 'Aachen', 'Western Europe', '52066', 'Germany', '0241-039123'), ('FRANK', 'Frankenversand', 'Peter Franken', 'München', 'Western Europe', '80805', 'Germany', '089-0877310'), ('KOENE', 'Königlich Essen', 'Philip Cramer', 'Brandenburg', 'Western Europe', '14776', 'Germany', '0555-09876'), ('LEHMS', 'Lehmanns Mark

**Aufbau**

Überführe mindestens fünf natürliche Sprachfragen in funktionierende SQL-Abfragen und prüfe die Ergebnisse manuell auf Korrektheit.

**✅ Erledigt wenn:** Jede der fünf Fragen erzeugt valides SQL ohne manuelle Korrektur — die Ergebnisse sind dokumentiert.

In [28]:
# Aufbau: 5 Fragen in SQL überführen und Ergebnisse dokumentieren
# Startpunkt: sql_generator + execute_query aus Kapitel 4/5

fragen = [
    "Welche Kunden haben mehr als 5 Bestellungen aufgegeben?",
    "Was sind die 5 teuersten Produkte im Sortiment?",
    "Welche Mitarbeiter wurden vor 1994 eingestellt?",
    "Wie hoch ist der durchschnittliche Bestellwert?",
    "Wie viele Mitarbeiter gibt es pro Land?",
]

for frage in fragen:
    result = sql_generator.invoke({"query": frage, "history_text": ""}, config=run_cfg)
    print(f"Frage:    {frage}")
    print(f"SQL:      {result.sql}")
    is_valid, msg = validate_sql(result.sql)
    print(f"Validierung: {msg}")
    if is_valid:
        print(f"Ergebnis: {execute_query(result.sql)}")
    print('---')


Frage:    Welche Kunden haben mehr als 5 Bestellungen aufgegeben?
SQL:      SELECT c.CustomerID,
       c.CompanyName,
       c.ContactName,
       COUNT(DISTINCT o.OrderID) AS OrdersCount
FROM Customers c
JOIN Orders o ON o.CustomerID = c.CustomerID
GROUP BY c.CustomerID, c.CompanyName, c.ContactName
HAVING COUNT(DISTINCT o.OrderID) > 5
ORDER BY OrdersCount DESC, c.CustomerID
LIMIT 10;
Validierung: OK
Ergebnis: [('BSBEV', "B's Beverages", 'Victoria Ashworth', 210), ('LILAS', 'LILA-Supermercado', 'Carlos González', 203), ('RICAR', 'Ricardo Adocicados', 'Janete Limeira', 203), ('GOURL', 'Gourmet Lanchonetes', 'André Fonseca', 202), ('PRINI', 'Princesa Isabel Vinhos', 'Isabel de Castro', 200), ('HUNGC', 'Hungry Coyote Import Store', 'Yoshi Latimer', 198), ('TORTU', 'Tortuga Restaurante', 'Miguel Angel Paolino', 197), ('ANATR', 'Ana Trujillo Emparedados y helados', 'Ana Trujillo', 195), ('FOLIG', 'Folies gourmandes', 'Martine Rancé', 195), ('RANCH', 'Rancho grande', 'Sergio Gutiérrez', 19

**Vertiefung**

Verbessere die Robustheit gegen unklare Anfragen, diskutiere SQL-Sicherheitsregeln oder lasse Query-Erklärungen im Klartext ausgeben.

**✅ Erledigt wenn:** Der Agent gibt bei einer unklaren Anfrage eine verständliche Fehlermeldung aus — kein ungültiges SQL, kein unbehandelter Fehler.

In [29]:
# Vertiefung: Robustheit und Sicherheit

# Option A: Unklare Anfrage testen — was gibt der Agent zurück?
# Option B: SQL-Sicherheitsregel im System-Prompt einbauen (nur SELECT erlaubt)
# Option C: Query-Erklärung im Klartext ausgeben lassen

# Test-Fragen für Robustheit:
unklare_fragen = [
    "Zeig mir alles",       # zu vage
    "Lösche alle Daten",    # gefährlich
    "xyz???",               # kein Sinn
]
# ...